Stratified Train / Val / Test Split (70 / 15 / 15)

In [ ]:
def create_splits(data_by_class, out_dir, train_size=0.70, val_size=0.15, seed=SEED, min_per_class=5):
    out_dir = Path(out_dir)
    skipped = []
    for cls, files in data_by_class.items():
        if len(files) < min_per_class:
            skipped.append((cls, len(files)))
            continue
        train_files, temp_files = train_test_split(files, train_size=train_size, random_state=seed)
        rel_val = val_size / (1 - train_size)
        val_files, test_files = train_test_split(temp_files, train_size=rel_val, random_state=seed)

        for split_name, split_files in [("train", train_files), ("val", val_files), ("test", test_files)]:
            dest = out_dir / split_name / cls
            dest.mkdir(parents=True, exist_ok=True)
            for f in split_files:
                shutil.copy2(f, dest / f.name)
        print(f"{cls:25s} train={len(train_files):4d} val={len(val_files):4d} test={len(test_files):4d}")

    if skipped:
        print("\nWARNING — classes skipped (too few images):")
        for name, n in skipped:
            print(f"  {name}: {n} images  (need >= {min_per_class})")
        print("Document any exclusion in the report's Data/Ethics section.")
    return skipped

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
skipped_classes = create_splits(data_by_class, SPLIT_DIR)

TRAIN_DIR = SPLIT_DIR / "train"
VAL_DIR = SPLIT_DIR / "val"
TEST_DIR = SPLIT_DIR / "test"


In [ ]:
BATCH_SIZE = 32

# ---- Generators for the from-scratch baseline CNN (expects [0,1] input) ----
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest",
)
eval_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True, seed=SEED,
)
val_generator = eval_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)
test_generator = eval_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)

CLASSES = list(train_generator.class_indices.keys())
NUM_CLASSES = len(CLASSES)
assert train_generator.class_indices == val_generator.class_indices == test_generator.class_indices, \
    "Class-index mismatch between splits — check for a missing/extra class folder in one split."
print(f"\nNum classes: {NUM_CLASSES}")
print(f"Train/Val/Test sizes: {train_generator.samples}/{val_generator.samples}/{test_generator.samples}")

transfer_train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest",
)  # NOTE: no rescale=1./255 here
transfer_eval_datagen = ImageDataGenerator()  # NOTE: no rescale here either

transfer_train_generator = transfer_train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True, seed=SEED,
)
transfer_val_generator = transfer_eval_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)
transfer_test_generator = transfer_eval_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False,
)
assert transfer_train_generator.class_indices == train_generator.class_indices, \
    "Transfer generator class-index mapping doesn't match the baseline generator."


In [ ]:
# Class weights, computed on the TRAIN split, to counter the imbalance found in Section 3 —
# an explicit, reportable bias-mitigation step for the Ethics section.
train_labels = train_generator.classes
class_weight_values = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_labels), y=train_labels
)
CLASS_WEIGHT_DICT = {i: w for i, w in enumerate(class_weight_values)}
print("Class weights (balanced):")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:20s} weight={CLASS_WEIGHT_DICT[i]:.3f}")
